# A more complex example: phase transition in Cu$_2$S

## Introduction

Let us come back to the example we were talking about in the beginning: Reproducing figure 8 from this [paper](https://doi.org/10.1021/acs.jctc.8b01092). In this notebook, we implement a very similar approach with the notable exception that we choose the settings in a way that the effect can be seen in a reasonable time frame. It is unlikely that we will meet the exact transition point as the simulation has to be converged properly. Thus, we will provide some better converged trajectory for more detailed analysis.

In [ ]:
# Equilibration
import ase
import ase.io
import ase.units
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.md.langevin import Langevin
from ase.md.npt import MelchionnaNPT
import numpy as np
from ase.io.trajectory import Trajectory
import time
import os
from mace.calculators import MACECalculator
from pathlib import Path
from ase.build import make_supercell
import torch

base_path = Path.cwd().parents[1]

seed = 1234
T_equil = 300
n_equil = 200  # In NVT ensemble
model_file_name = (
    base_path
    / "data"
    / "models"
    / "full_dataset"
    / "M_2_ell_2_16_16_cut_5"
    / "Cu2S_stagetwo.model"
)
struct_file_name = base_path / "data" / "Cu2S_monoclinic" / "Cu2S_P21C.extxyz"
dtype = "float32"
dt = 2
pfactor = 90  # Rough bulk modulus for the Barostat in GPa
ttime = (
    20 * dt
)  # Thermostat characteristic time scale (deliberately lower than usual)
ptime = 100 * dt  # Barostat characteristic time scale
pressure = 1.5  # Pressure in GPa


atoms = ase.io.read(struct_file_name)
atoms = make_supercell(atoms, np.diag([2, 2, 2]))

if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"


calc = MACECalculator(model_file_name, device=device, default_dtype=dtype)

rng = np.random.default_rng(seed=seed)
MaxwellBoltzmannDistribution(atoms, temperature_K=float(T_equil), rng=rng)
atoms.calc = calc

integrator = Langevin(
    atoms,
    dt * ase.units.fs,
    temperature_K=float(T_equil),
    friction=1
    / ttime
    / ase.units.fs
    * 4,  # we want to reach the target temperature FAST
    rng=rng,
)

densities = []
temperatures = []
volumes = []
pressures = []


def print_md_info(current_atoms=atoms, current_integrator: MelchionnaNPT = integrator):
    n_atoms = len(current_atoms)
    e_pot_per_atom = current_atoms.get_potential_energy() / n_atoms
    e_kin_per_atom = current_atoms.get_kinetic_energy() / n_atoms
    kinetic_temperature = e_kin_per_atom / (1.5 * ase.units.kB)
    density = np.sum(current_atoms.get_masses()) / current_atoms.get_volume()
    stress_voigt = current_atoms.get_stress(voigt=True)
    P_virial = -np.mean(stress_voigt[:3])
    mv2 = (
        current_atoms.get_masses()[:, np.newaxis]
        * current_atoms.get_velocities() ** 2
    )
    P_kinetic = np.sum(mv2) / (3 * current_atoms.get_volume())
    pressure = (P_kinetic + P_virial) / ase.units.GPa
    densities.append(density)
    temperatures.append(kinetic_temperature)
    volumes.append(current_atoms.get_volume())
    pressures.append(pressure)
    print(
        f"Step #{current_integrator.nsteps + 1}: "
        f"Epot = {e_pot_per_atom} eV / atom, T = {kinetic_temperature} K, "
        f"E = {e_pot_per_atom + e_kin_per_atom} eV / atom, "
        f"Density = {density} amu / A^3",
        f"Pressure = {pressure} GPa",
        flush=True,
    )


integrator.attach(print_md_info, interval=1)

t1 = time.time()
integrator.run(n_equil)
t_equil = time.time() - t1
print("equilibration time:", t_equil, flush=True)
atoms_equil = atoms.copy()

After the equilibration at our initial temperature, we run the primary simulation. The way we implemented it here is that we increase the target temperature of the thermostat every 100 steps until we reach the final temperature. It is of course better to continuously increase the temperature, but we are reducing computational overhead this way. 

In [ ]:
n_run = 500
T_ramp_steps = 10
# we start from the previous temperature
T_start = atoms.get_temperature()
# here, we specify by how much we increase it
T_end = atoms.get_temperature() + 50


traj_path = (
    base_path
    / "data"
    / "MD_trajectories"
    / f"Cu2S_P21C_222_NPT_Tramp_{T_start:.0f}_{T_end:.0f}K.traj"
)
print("trajectory path:", traj_path)
if traj_path.exists():
    traj_path.unlink()
log_file_path = (
    base_path
    / "data"
    / "MD_trajectories"
    / f"Cu2S_P21C_222_NPT_Tramp_{T_start:.0f}_{T_end:.0f}K.log"
)

t1 = time.time()
num_t_steps = n_run // T_ramp_steps
T_intervals = np.linspace(T_start, T_end, num_t_steps)
for T in T_intervals:
    dyn = MelchionnaNPT(
        atoms,
        dt * ase.units.fs,
        temperature_K=T,
        externalstress=pressure * ase.units.GPa,
        ttime=ttime * ase.units.fs,
        pfactor=pfactor * ase.units.GPa * (ptime * ase.units.fs) ** 2,
        logfile=log_file_path,
        trajectory=traj_path,
        loginterval=10,
        append_trajectory=True,
    )

    dyn.attach(print_md_info, interval=1, current_integrator=dyn)

    dyn.run(T_ramp_steps)
t_run = time.time() - t1
print("run time:", t_run, flush=True)

Here, we will plot the key metrics:
Does the run align with the expectations? When does the structure reduce itself to volumes as low as in the paper? Try running the above script again to reach higher temperatures. 

What happens if you ramp up the temperature faster?

In [ ]:
import matplotlib.pyplot as plt

kg_2_amu = 6.0229552894949e26
m_2_Angstrom = 1e-10
plt.figure(1)
plt.plot(pressures)
plt.xlabel("time / fs")
plt.ylabel("pressure / GPa")
plt.tight_layout()
plt.figure(2)
plt.plot(temperatures)
plt.xlabel("time / fs")
plt.ylabel("temperature / K")
plt.tight_layout()
plt.figure(3)
plt.plot(np.asarray(densities) / kg_2_amu / m_2_Angstrom**3)
plt.xlabel("time / fs")
plt.ylabel(r"density / kg$\,\mathrm{m}^{-3}$")
plt.tight_layout()
plt.figure(4)
plt.plot(volumes)
plt.xlabel("time / fs")
plt.ylabel(r"volume / $\mathrm{\AA}^3$")
plt.tight_layout()
plt.show()

It is clear that it is difficult to obtain the correct phase transition while still maintaining a low computational provile. Depending on the settings it might be too gradual to notice or at a different temperature. This is why it is very important to properly converge the settings. You can find a trajectory ran for 1 ns in `data/NPT_long_trajectory.tar.gz`. Note, that it only contains 200 equidistant time steps

TASK:
- visualize the metrics above using the provided trajectory file and compare to figure 8 in the paper
- Plot the MSDs as a function of temperature and compare to figure 9 in the paper

In [ ]:
# solution for Cu2S temp ramp example (make sure the path of the trajectory is correct):
from ase.io.trajectory import Trajectory
from ase.visualize import view
import ase.units 
fname = base_path/"data"/"NPT_split.traj"

traj = Trajectory(fname, "r")
view(traj)

import numpy as np
import matplotlib.pyplot as plt

kg_2_amu = 6.0229552894949e26
m_2_Angstrom = 1e-10
volumes = []
temperatures = []
densities = []
pressures = []
pos_array = []
for atoms in traj:
    n_atoms = len(atoms)
    e_pot_per_atom = atoms.get_potential_energy() / n_atoms
    e_kin_per_atom = atoms.get_kinetic_energy() / n_atoms
    kinetic_temperature = e_kin_per_atom / (1.5 * ase.units.kB)
    density = np.sum(atoms.get_masses()) / atoms.get_volume()
    stress_voigt = atoms.get_stress(voigt=True)
    P_virial = -np.mean(stress_voigt[:3])
    mv2 = atoms.get_masses()[:, np.newaxis] * atoms.get_velocities() ** 2
    P_kinetic = np.sum(mv2) / (3 * atoms.get_volume())
    pressure = (P_kinetic + P_virial) / ase.units.GPa
    densities.append(density)
    temperatures.append(kinetic_temperature)
    volumes.append(atoms.get_volume())
    pressures.append(pressure)
    pos_array.append(atoms.positions)

pos_array = np.array(pos_array)
indices = np.where(atoms.symbols == "Cu")[0]
mean_pos = np.mean(pos_array, axis=0)
MSDs_Cu = np.mean(
    np.linalg.norm(pos_array[:, indices] - mean_pos[indices], axis=2) ** 2,
    axis=1,
)

# total time of 1 ns
dt = 1e6 / len(traj)
timearr = np.arange(len(traj)) * dt
plt.figure(1)
plt.plot(timearr, pressures)
plt.xlabel("time / fs")
plt.ylabel("pressure / GPa")
plt.tight_layout()
plt.figure(2)
plt.plot(timearr, temperatures)
plt.xlabel("time / fs")
plt.ylabel("temperature / K")
plt.tight_layout()
plt.figure(3)
plt.plot(timearr, np.asarray(densities) / kg_2_amu / m_2_Angstrom**3)
plt.xlabel("time / fs")
plt.ylabel(r"density / kg$\,\mathrm{m}^{-3}$")
plt.tight_layout()
plt.figure(4)
plt.plot(timearr, volumes)
plt.xlabel("time / fs")
plt.ylabel(r"volume / $\mathrm{\AA}^3$")
plt.tight_layout()
plt.figure(5)
plt.plot(timearr, MSDs_Cu)
plt.xlabel("time / fs")
plt.ylabel(r"MSD / $\mathrm{\AA}^2$")
plt.tight_layout()
plt.show()